In [4]:
import numpy as np

euler_cube = np.load('./output/euler_cube.npy')
coord_cube = np.load('./output/coord_cube.npy')
Nb_cube = np.load('./output/Nb_cube.npy')
Sn_cube = np.load('./output/Sn_cube.npy')

In [5]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.transform import Rotation


euler_points = euler_cube.reshape(euler_cube.shape[0]*euler_cube.shape[1]*euler_cube.shape[2],euler_cube.shape[3])



coordinates = coord_cube.reshape(coord_cube.shape[0]*coord_cube.shape[1]*coord_cube.shape[2],coord_cube.shape[3])


#EBSD_points = np.append(coordinates,euler_points,axis=-1)

nonzero = list(not(np.array_equal(euler_points[i,:],np.array((0.0,0.0,0.0)))) for i in range(euler_points.shape[0]))
coordinates = coordinates[nonzero]
euler_points = euler_points[nonzero]

rotations = Rotation.from_euler('XZX',euler_points)

del euler_points

#coordinates = np.append(coordinates,[[100,100,100]],axis=0)
np.random.seed(0)
subsample = np.random.randint(0,coordinates.shape[0],size=50000)

# rotations = rotations[subsample]
# coordinates = coordinates[subsample]

#plt.scatter(euler_points[:,0],euler_points[:,1],c=euler_points[:,2])

In [6]:
from scipy.spatial import Voronoi

vor = Voronoi(coordinates)

In [7]:
from scipy.sparse import csr_array

vertices = vor.vertices

infinite_point = np.array((100.0,100.0,100.0))
vertices = np.concatenate((vertices,infinite_point[None,:]),axis=0)

edges = []
ridge_edges = []
i = 0

for ridge in vor.ridge_vertices:
    new_ridge = []
    for (vert1, vert2) in zip(ridge,ridge[1:]+ridge[:1]):
        """create new edge"""
        new_edge = [vert1,vert2]
        edges.append(new_edge)

        """add edge to ridge"""
        new_ridge.append(i)
        i += 1

    ridge_edges.append(new_ridge)

edges, inverse = np.unique(np.array(edges),axis=0,return_inverse=True)
ridge_edges = [[inverse[j] for j in ridge_edges[i]] for i in range(len(ridge_edges))]







In [8]:
region_ridges = [[] for _ in range(len(vor.regions))]

for ridge_ind, ridge in enumerate(vor.point_region[vor.ridge_points]):
    for region_ind in ridge:
        region_ridges[region_ind].append(ridge_ind)

In [9]:
T_VE = csr_array((vor.vertices.shape[0],edges.shape[0]),dtype='bool')
T_EF = csr_array((edges.shape[0],len(ridge_edges)),dtype='bool')
T_FD = csr_array((len(ridge_edges),len(vor.regions)),dtype='bool')

In [10]:
X,Y = np.indices(edges.shape)

T_VE[edges.flatten(),X.flatten()] = True

# for i, edge in enumerate(edges):
#     T_VE[edge[0],i] = True
#     T_VE[edge[1],i] = True

/home/vike/dev/3debsd/.venv/lib/python3.13/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_array is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


In [11]:
ridge_ind = np.repeat(np.arange(len(ridge_edges)),[len(ridge_edges[i]) for i in range(len(ridge_edges))])
edge_ind = [edge for ridge in ridge_edges for edge in ridge]

T_EF[edge_ind,ridge_ind] = True

# for ridge_ind, ridge in enumerate(ridge_edges):
#     for edge in ridge:
#         T_EF[edge,ridge_ind] = True

In [12]:
ridge_ind = [ridge for region in region_ridges for ridge in region]
region_ind = np.repeat(np.arange(len(region_ridges)),[len(region_ridges[i]) for i in range(len(region_ridges))])

T_FD[ridge_ind,region_ind] = True

# for face in np.arange(0,vor.ridge_points.shape[0]):
#     for domain in vor.point_region[vor.ridge_points[face,:]]:
#         T_FD[face,domain] = True

In [14]:
"""Find the vacuum interface"""
from skimage.measure import marching_cubes
import pyvista as pv

total_xray_cube = Nb_cube + Sn_cube

# plt.hist(total_xray_cube.flatten())

verts, faces, normals, values = marching_cubes(total_xray_cube,120000,spacing=(0.1,0.1,0.1))

surface_mesh = pv.PolyData.from_regular_faces(verts, faces)

In [15]:
def save_obj(filepath, verts, faces):
    file = open(filepath, 'w')
    for vertex in verts:
        file.write(f"v {vertex[0]} {vertex[1]} {vertex[2]}\n")

    for face in faces:
        file.write(f"f ")
        for ind in face:
            file.write(f"{ind+1} ")
        file.write("\n")  

    file.close()

In [16]:
save_obj('output/sample_surface.obj',verts,faces)

In [17]:
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# # Display resulting triangular mesh using Matplotlib. This can also be done
# # with mayavi (see skimage.measure.marching_cubes docstring).
# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(111, projection='3d')

# # Fancy indexing: `verts[faces]` to generate a collection of triangles
# mesh = Poly3DCollection(verts[faces])
# mesh.set_edgecolor('k')
# ax.add_collection3d(mesh)

# ax.set_xlabel("x-axis: a = 6 per ellipsoid")
# ax.set_ylabel("y-axis: b = 10")
# ax.set_zlabel("z-axis: c = 16")

# ax.set_xlim(0, 10)  # a = 6 (times two for 2nd ellipsoid)
# ax.set_ylim(0, 10)  # b = 10
# ax.set_zlim(0, 10)  # c = 16

# plt.tight_layout()
# plt.show()

In [18]:
"""add bounding box"""

'add bounding box'

In [19]:
"""calculate intersections between domain faces and bounding surface"""

'calculate intersections between domain faces and bounding surface'

In [20]:
"""create new faces and update intersecting domains"""

'create new faces and update intersecting domains'

In [21]:
misorientations = rotations[vor.ridge_points[:,0]] * rotations[vor.ridge_points[:,1]].inv()
misorientations = np.linalg.norm(misorientations.as_rotvec(),axis=-1)

# GBs = [i for (i,v) in zip(vor.ridge_points,(misorientations > 0.1)) if v]
# nonGBs = [i for (i,v) in zip(vor.ridge_points,(misorientations < 0.1)) if v]

# GBs = vor.point_region[vor.ridge_points[misorientations > 0.1]]
# nonGBs = vor.point_region[vor.ridge_points[misorientations < 0.1]]

GBs = np.arange(0,vor.ridge_points.shape[0])[misorientations > 0.1]
nonGBs = np.arange(0,vor.ridge_points.shape[0])[misorientations <= 0.1]

In [22]:
A_DGB = csr_array((T_FD.shape[1],T_FD.shape[1]))
T_FDGB = csr_array(T_FD.shape)

T_FDGB = T_FD * (misorientations > 0.05)[:,None]
T_FDGB.eliminate_zeros()

A_DGB = T_FDGB.T @ T_FDGB
A_DGB.eliminate_zeros()
A_D = T_FD.T @ T_FD
A_D.eliminate_zeros()
A_DnonGB = A_D - A_DGB
A_DnonGB.eliminate_zeros()

In [23]:
from scipy.sparse.csgraph import breadth_first_tree

# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(111)


remaining_regions = vor.point_region
grains = []

while remaining_regions.shape[0] > 0:
    csr_grain = breadth_first_tree(A_DnonGB,remaining_regions[0],directed=False)

    csr_grain = csr_grain.tocoo()

    grain_regions = np.unique(np.concatenate((csr_grain.coords[0],csr_grain.coords[1],remaining_regions[0,None])))

    remaining_regions = np.setdiff1d(remaining_regions,grain_regions,assume_unique=True)

    grains.append(grain_regions)

    # grain_ids = [vor.regions[v] for v in grain_regions]
    # color = np.random.rand(3,)

    # ax = fig.axes[0]

    # for i, regionid in enumerate(grain_ids):

    #     # print(f"point index: {i}")
    #     # print(f"region ID: {regionid}")
        
    #     if -1 not in regionid:

    #         vertices = vor.vertices[regionid]
    #         vertices = np.asarray(vertices)
    #         ax.fill(vertices[:,0],vertices[:,1],facecolor=color)


In [24]:
T_PD = csr_array((len(vor.point_region),len(vor.regions)),dtype='bool')

point_ind = np.arange((len(vor.point_region)))
domain_ind = vor.point_region

T_PD[point_ind,domain_ind] = True


/home/vike/dev/3debsd/.venv/lib/python3.13/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_array is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


In [25]:
T_DG = csr_array((len(vor.regions),len(grains)),dtype='bool')

grain_ind, domain_ind = np.array([[i,domain] for i, grain in enumerate(grains) for domain in grain]).T

T_DG[domain_ind,grain_ind] = True

# for i, grain in enumerate(grains):
#     for domain in grain:
#         T_DG[domain,i] = True

In [26]:
B_D = T_DG @ T_DG.T
B_D.eliminate_zeros()

In [27]:
T_FG = csr_array((T_FD.shape[0],T_DG.shape[1]),dtype='bool')

T_FG = T_FD@(B_D * A_DGB)@T_DG * (T_FD@T_DG)
T_FG.eliminate_zeros()

In [28]:
GB_vertices = [vor.ridge_vertices[face] if -1 not in vor.ridge_vertices[face] else [] for face in T_FG[:,[1]].tocoo().coords[0]]

In [29]:
GB_vert_coords = [vertices[GB_vertices[face]] for face in range(len(GB_vertices))]

In [30]:
# for face in GB_vert_coords:
#     for vert in face:
#         if np.any(vert > 10.0):
#             GB_vert_coords.remove(face)

In [31]:
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection
# from copy import deepcopy

# # Display resulting triangular mesh using Matplotlib. This can also be done
# # with mayavi (see skimage.measure.marching_cubes docstring).
# fig = plt.figure(figsize=(10, 10))
# ax = fig.add_subplot(111, projection='3d')

# for i in range(5):
#     GB_vertices = [vor.ridge_vertices[face] if -1 not in vor.ridge_vertices[face] else [] for face in T_FG[:,[i]].tocoo().coords[0]]
#     GB_vert_coords = [vertices[GB_vertices[face]] for face in range(len(GB_vertices))]
#     # temp = []
#     # for i, face in enumerate(GB_vert_coords):
#     #     for vert in face:
#     #         if np.any((vert < 10.0)) and np.any((vert > 0.0)):
#     #             temp.append(face)

#     # GB_vert_coords = temp

#     # Fancy indexing: `verts[faces]` to generate a collection of triangles
#     mesh = Poly3DCollection(GB_vert_coords)
    
#     color = np.random.rand((3))
#     mesh.set_color(color)
#     mesh.set_edgecolor('k')
#     ax.add_collection3d(mesh)

# ax.set_xlabel("x-axis: a = 6 per ellipsoid")
# ax.set_ylabel("y-axis: b = 10")
# ax.set_zlabel("z-axis: c = 16")

# ax.set_xlim(0, 10)  # a = 6 (times two for 2nd ellipsoid)
# ax.set_ylim(0, 10)  # b = 10
# ax.set_zlim(0, 10)  # c = 16

# plt.tight_layout()
# plt.show()

In [32]:
# GB_vertices = [vor.ridge_vertices[face] for face in T_FG[:,[3]].tocoo().coords[0]]

In [33]:
# # Create a new list without out-of-bounds vertices
# filtered_GB_vertices = []

# for face in GB_vertices:
#     # Check if any vertex is -1 (indicating infinity in Voronoi diagrams)
#     if -1 in face:
#         continue
        
#     # Check if any vertex is out of bounds
#     verts_coords = vertices[face]
#     if verts_coords.max() > 10.0 or verts_coords.min() < 0.0:
#         continue
        
#     # If we get here, the face is valid
#     filtered_GB_vertices.append(face)

# print(f"Filtered out {len(GB_vertices) - len(filtered_GB_vertices)} faces with out-of-bounds vertices")

# # Replace the original list with the filtered one
# GB_vertices = filtered_GB_vertices



In [34]:
# # First, identify which vertices are actually used in GB_vertices
# used_vertices = set()
# for triangle in GB_vertices:
#     for idx in triangle:
#         used_vertices.add(idx)

# # Create a mapping from old indices to new indices
# old_to_new = {}
# new_vertices = []

# for i, idx in enumerate(sorted(used_vertices)):
#     old_to_new[idx] = i
#     new_vertices.append(vertices[idx])

# # Convert vertices to numpy array
# new_vertices = np.array(new_vertices)

# # Update the indices in GB_vertices
# new_GB_vertices = []
# for triangle in GB_vertices:
#     new_triangle = [old_to_new[idx] for idx in triangle]
#     new_GB_vertices.append(new_triangle)

# print(f"Reduced vertices from {len(vertices)} to {len(new_vertices)}")

In [35]:
def plot_grain(id):
    GB_vertices = [vor.ridge_vertices[face] for face in T_FG[:,[id]].tocoo().coords[0]]

    # Create a new list without out-of-bounds vertices
    filtered_GB_vertices = []

    for face in GB_vertices:
        # Check if any vertex is -1 (indicating infinity in Voronoi diagrams)
        if -1 in face:
            continue
            
        # # Check if any vertex is out of bounds
        # verts_coords = vertices[face]
        # if verts_coords.max() > 10.0 or verts_coords.min() < 0.0:
        #     continue
            
        # If we get here, the face is valid
        filtered_GB_vertices.append(face)

    print(f"Filtered out {len(GB_vertices) - len(filtered_GB_vertices)} faces with out-of-bounds vertices")

    # Replace the original list with the filtered one
    GB_vertices = filtered_GB_vertices

    # First, identify which vertices are actually used in GB_vertices
    used_vertices = set()
    for triangle in GB_vertices:
        for idx in triangle:
            used_vertices.add(idx)

    # Create a mapping from old indices to new indices
    old_to_new = {}
    new_vertices = []

    for i, idx in enumerate(sorted(used_vertices)):
        old_to_new[idx] = i
        new_vertices.append(vertices[idx])

    # Convert vertices to numpy array
    new_vertices = np.array(new_vertices)

    # Update the indices in GB_vertices
    new_GB_vertices = []
    for triangle in GB_vertices:
        new_triangle = [old_to_new[idx] for idx in triangle]
        new_GB_vertices.append(new_triangle)

    print(f"Reduced vertices from {len(vertices)} to {len(new_vertices)}")


    # roi = pv.Box(bounding_box, level=100).clip_surface(surface_mesh, invert=False)
    # roi += surface_mesh.clip_surface(roi, invert=True)

    bounding_box = [0.2, coordinates[:,0].max()-1.2, 0.2, coordinates[:,1].max()-0.2, 0.2, coordinates[:,2].max()-0.2]
    roi = pv.Box(bounding_box, 4, quads=False)
    # roi = surface_mesh.extrude_trim([1,0,0], pv.Plane(i_size=50, j_size=50, direction=[1,0,0]))



    # polydata = pv.PolyData.from_irregular_faces(new_vertices, new_GB_vertices).clip_box(bounding_box, invert=False).clip_surface(surface_mesh, invert=False)
    # polydata += surface_mesh.clip_surface(polydata, invert=False)

    grain_mesh = pv.PolyData.from_irregular_faces(new_vertices, new_GB_vertices).compute_normals().triangulate()  
    # polydata = grain_mesh.clip_box(bounding_box, invert=False)
    polydata = grain_mesh.boolean_intersection(roi)

    return polydata
    

In [36]:
import pyvista as pv
pv.set_jupyter_backend('client')
pv.global_theme.allow_empty_mesh = True


# plot_mesh = pv.PolyData.from_irregular_faces(new_vertices, new_GB_vertices)
# plot_mesh.plot(cpos='xy', show_edges=True)

plotter = pv.Plotter()

colors = np.random.rand(len(grains),3)



for id in range(len(grains)):
# for id in [0,1,2,3]:
    plotter.add_mesh(plot_grain(id), color=colors[id])


plotter.show()

ImportError: Please install trame dependencies: pip install "pyvista[jupyter]"

In [ ]:
centroid = np.average(coordinates, axis=0)

def save_grain(id):
    GB_vertices = [vor.ridge_vertices[face] for face in T_FG[:,[id]].tocoo().coords[0]]
    vertices_copy = vertices.copy()

    print(f"{vertices.shape}")

    # Create a new list without out-of-bounds vertices
    filtered_GB_vertices = []

    for face in GB_vertices:
        # Check if any vertex is -1 (indicating infinity in Voronoi diagrams)
        for i, vertex in enumerate(face):
            if vertex == -1:
                new_vertex = 100 * (vertices[face[i-1]] - centroid)
                vertices_copy = np.concat((vertices_copy, new_vertex[None,:]), axis = 0)
                face[i] = vertices_copy.shape[0] - 1
            
        # # Check if any vertex is out of bounds
        # verts_coords = vertices[face]
        # if verts_coords.max() > 10.0 or verts_coords.min() < 0.0:
        #     continue
            
        # If we get here, the face is valid
        filtered_GB_vertices.append(face)
    
    print(f"{vertices_copy.shape}")

    print(f"Filtered out {len(GB_vertices) - len(filtered_GB_vertices)} faces with out-of-bounds vertices")

    # Replace the original list with the filtered one
    GB_vertices = filtered_GB_vertices

    # First, identify which vertices are actually used in GB_vertices
    used_vertices = set()
    for triangle in GB_vertices:
        for idx in triangle:
            used_vertices.add(idx)

    # Create a mapping from old indices to new indices
    old_to_new = {}
    new_vertices = []

    for i, idx in enumerate(sorted(used_vertices)):
        old_to_new[idx] = i
        new_vertices.append(vertices_copy[idx])

    # Convert vertices to numpy array
    new_vertices = np.array(new_vertices)

    # Update the indices in GB_vertices
    new_GB_vertices = []
    for triangle in GB_vertices:
        new_triangle = [old_to_new[idx] for idx in triangle]
        new_GB_vertices.append(new_triangle)

    print(f"Reduced vertices from {len(vertices)} to {len(new_vertices)}")


    save_obj(f'output/grain{i}.obj',new_vertices,new_GB_vertices)
    


GB_vertices = []
GB_vert_coords = []
for i in range(T_FG.shape[1]):
    if T_DG[:,[i]].tocoo().coords[0].shape[0] > 100:
        # for face in T_FG[:,[i]].tocoo().coords[0]:
        #     if -1 not in vor.ridge_vertices[face]:
        #         GB_vertices.extend(vor.ridge_vertices[face])
        #     else:
        #         ridge_vertices = vor.ridge_vertices[face].copy()
                

        # for face in range(len(GB_vertices)):
        #     GB_vert_coords.extend(vertices[GB_vertices[face]])

        # # GB_vertices = [vor.ridge_vertices[face] if -1 not in vor.ridge_vertices[face] else [] for face in T_FG[:,[i]].tocoo().coords[0]]
        # # GB_vert_coords = [vertices[GB_vertices[face]] for face in range(len(GB_vertices))]

        # faces = GB_vertices
        # verts = vertices

        # save_obj(f'output/grain{i}.obj',verts,faces)

        save_grain(i)

Filtered out 0 faces with out-of-bounds vertices
Reduced vertices from 221701 to 26350
Filtered out 0 faces with out-of-bounds vertices


IndexError: index 222524 is out of bounds for axis 0 with size 222053

In [ ]:
import pyvista as pv

def unstructured_grid_from_grain(id):
    domains = T_DG[:,[id]].nonzero()[0]
    T_VD = T_VE @ T_EF @ T_FD
    cell_types = [pv.CellType.POLYHEDRON] * len(domains)
    
    used_vertices = set()
    cell_point_ids = []

    for domain in domains:
        if np.any(domain == -1):
            continue
        vertex_indices = T_VD[:,domain].nonzero()[0]
        used_vertices.update(vertex_indices)

        cell_point_ids.append(vertex_indices)


    index_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted(used_vertices))}

    cell_array = []
    for cell in cell_point_ids:
        new_cell = [index_map[pt] for pt in cell]
        cell_array.append(len(new_cell))
        cell_array.extend(new_cell)

    cells = cell_array
    points = vertices[sorted(used_vertices)]

    bounding_box = [0, coordinates[:,0].max(), 0, coordinates[:,1].max(), 0, coordinates[:,2].max()]
    roi = pv.Box(bounds=bounding_box)
    grid = pv.UnstructuredGrid(cells, cell_types, points)

    return grid, roi


In [ ]:
# import pyvista as pv
# pv.set_jupyter_backend('static')
# pv.global_theme.allow_empty_mesh = True

# plotter = pv.Plotter()

# colors = np.random.rand(len(grains),3)
# # for id in range(len(grains)):
# for id in [1,]:
#     grain_grid, roi = unstructured_grid_from_grain(id)
#     plotter.add_mesh(grain_grid, color=colors[id], show_edges=True)
#     plotter.add_mesh(roi, color='gray', opacity=0.5, show_edges=True)

# plotter.show()

In [ ]:
from scipy.spatial import KDTree

tree = KDTree(coordinates)
query_points = coord_cube.reshape(coord_cube.shape[0]*coord_cube.shape[1]*coord_cube.shape[2],coord_cube.shape[3])

dd, ii = tree.query(query_points)

In [ ]:
query_point_ids = np.arange(ii.shape[0])
query_domain_ids = vor.point_region[ii]


T_QD = csr_array((query_points.shape[0],len(vor.regions)),dtype='bool')
T_QD[query_point_ids,query_domain_ids] = True

In [ ]:
T_QG = T_QD @ T_DG

In [ ]:
import sys
import importlib
import plots3d
importlib.reload(sys.modules['plots3d'])
from plots3d import Scatter3D



Scatter3D(query_points[T_QG[:,3].coords[0]])

In [ ]:
def surf_distance(query_points, vertices, edges, faces):
    """Calculate the unsigned distance from the query points to a mesh using a kdtree approach.
    
    Parameters:
    -----------
    query_points : (N,3) array
        The coordinates of N points to calculate the distance to the mesh from.
    vertices : (M,3) array
        The coordinates of the M vertices that define the mesh.
    edges : (M,E) matrix of boolean
        A matrix defining which vertices constitute which edges.
    faces : (E,F) matrix of boolean
        A matrix defining which edges constitute which faces.

    Returns:
    --------
    distances : (N) array
        The closest distance between each of N points to the mesh.
    """


    search_tree = KDTree(vertices)

    # To first approximation, the closest distance to the 
    # mesh is the distance to the nearest vertex of the mesh.
    dd, ii = search_tree.query(query_points) 

    distances = dd

    return distances
    

In [ ]:
surf_distance(query_points[T_QG[:,3].coords[0]],)